In [ ]:
# Text generation using transformers (model GPT2) --> Hugging Face
from transformers import pipeline

pipe = pipeline("text-generation", model="openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
output = pipe("what is Machine learning")
print(output,"generated text")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'what is Machine learning for?"\n\nThe challenge is finding the right neural network to do this for you.\n\nI\'ve been playing with Machine learning all my life. It\'s one of those things that really has to be figured out, but I don\'t think it\'s going to be easy. I think it\'s going to take a long time to figure out.\n\nI\'ve been working on it for a couple of years now, and I\'ve been playing with it for a couple of years now, and I\'ve been playing with it for a couple of years now, and the work that I\'ve done is really fascinating to me.\n\nSo how does it work?\n\nIt\'s quite simple. You have two layers of neural networks in your brain. And a third layer, called a layer 1 network, is very specific about what you\'re looking at. And there\'s a third layer, called layer 2, which is more general than the second layer.\n\nAnd that layer is essentially just the layer that\'s going to be interested in what\'s going on inside the neural network.\n\nSo for example, if

In [ ]:
# making model next text generation probability using different approaches
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
gptmodel = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", device_map="auto")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# making text to token then by providing to model make prob predictions..
sentense = "we should create peacefull and"
token = tokenizer(sentense, return_tensors="pt").input_ids
output = gptmodel(token).logits[0,-1]
prob = torch.argmax(output)
print(tokenizer.decode(prob))

 peaceful


In [ ]:
# some important functions

# -----------------1. Greedy-probability------------------
# step1. get logits
# step2. find maximum prob percentage(argmax)
def greedy_prob(logits):
  return torch.argmax(logits,dim=-1)

# ----------------2. Top-k probability-------------------
# step1. get logits
# step2. get k words by (torch.topk(logits,k=may be any else number))
# step3. normalize(softmax) and sample(multinomial)
def top_k_prob(logits,k=50):
  value ,indexes = torch.topk(logits,k)
  prob = F.softmax(value,dim=-1)
  sample = torch.multinomial(prob,1)
  return indexes[sample]

# ---------------3. Top-p probability-------------------
# step1. get logits
# step2. logits to sort and nomalize(softmax)
# step3. calulate cumulative sum and should be nearest larger then p
# step4. normalize(softmax) and sample(multinomial)
def top_p_prob(logits,p=0.85):
  sorted_logits , sorted_indices = torch.sort(logits,descending=True)
  sorted_prob = F.softmax(sorted_logits,dim=-1)
  cumultive_sum = sorted_prob.cumsum(dim=-1)
  mask = cumultive_sum > p
  cumultive_sum[mask] - float("-inf")
  final_prob = F.softmax(cumultive_sum,dim=-1)
  sample = torch.multinomial(final_prob,1)
  return sorted_indices[sample]

#--------------- 4. Temprature probabillity-------------------
# step1. get logits
# step2. devide logit to temprature
# step3. normalize(softmax) and sample(multinomial)
def temprature_prob(logits,temprature=1.0):
  value = logits/temprature
  prob = F.softmax(value,dim=-1)
  return torch.multinomial(prob,1)

#-------------- 5. Random probability-------------------
# step1. get logits
# step2. nomalize(softmax) and sample(multinomial)
def random_prob(logits):
  prob = F.softmax(logits,dim=-1)
  return torch.multinomial(prob,1)


In [ ]:
# comparative view of five transformer text (word) probability(prob) approaches
print(f"Predictions by Greedy approach:", tokenizer.decode(greedy_prob(output)))
print(f"Predictions by Top-k approach:", tokenizer.decode(top_k_prob(output)))
print(f"Predictions by Top-p approach:", tokenizer.decode(top_p_prob(output)))
print(f"Predictions by Temprature approach:", tokenizer.decode(temprature_prob(output)))
print(f"Predictions by Random approach:", tokenizer.decode(random_prob(output)))


Predictions by Greedy approach:  peaceful
Predictions by Top-k approach:  tranquil
Predictions by Top-p approach:  swollen
Predictions by Temprature approach:  peaceful
Predictions by Random approach:  prosperous


In [ ]:
# probibility words with their percentage
sentense = "what is the supervised learning"
token = tokenizer(sentense, return_tensors="pt").input_ids
output = gptmodel(token).logits[0,-1]
prob_output = torch.softmax(output,dim=-1)

top10 = torch.topk(prob_output,k=10)

for index, values in zip(top10.indices,top10.values):  # indices gives word that in numeric form which a machine understand
                                                       # values give prob occuring percentage
                                                       # zip joins both indices and values
  print(f"{tokenizer.decode(index)} ----{values:.1%}")

 model ----15.1%
 of ----6.8%
 process ----5.3%
 approach ----4.3%
 system ----3.9%
 method ----3.7%
 algorithm ----3.6%
 problem ----2.0%
 environment ----1.8%
 that ----1.7%


## Sentiment Analysis

In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
type(ds)
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
print(ds['train'][0])


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
import pandas as pd

In [29]:
ds['train'].to_pandas()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [33]:
my_dataset_df = ds['train'].to_pandas()

In [35]:
my_dataset_df['text']
len(my_dataset_df['text'])

25000

In [36]:
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")